In [1]:
import numpy as np
import pandas as pd

In [37]:

fake_data = pd.DataFrame({'user_id':np.arange(1,101),'group':np.random.randint(0,2, size=100),'bookings':np.random.binomial(n=1, p=0.3, size=100),'sessions':np.random.randint(1, 10, size=100)})

fake_data['group'] = fake_data['group'].map({0: 'control', 1: 'treatment'})

In [38]:
fake_data

,user_id,group,bookings,sessions
0,1,control,0,5
1,2,control,1,9
2,3,treatment,1,7
3,4,control,0,5
4,5,treatment,1,8
...,...,...,...,...
95,96,treatment,0,7
96,97,treatment,1,2
97,98,control,0,9
98,99,control,0,7


In [42]:
fake_data['conversion_rate'] = fake_data['bookings']/fake_data['sessions']

fake_data

,user_id,group,bookings,sessions,conversion_rate
0,1,control,0,5,0.000000
1,2,control,1,9,0.111111
2,3,treatment,1,7,0.142857
3,4,control,0,5,0.000000
4,5,treatment,1,8,0.125000
...,...,...,...,...,...
95,96,treatment,0,7,0.000000
96,97,treatment,1,2,0.500000
97,98,control,0,9,0.000000
98,99,control,0,7,0.000000


In [34]:
group_stats = fake_data.groupby('group').agg(total_bookings =('bookings', 'sum'),
                                             total_sessions=('sessions', 'sum'))
group_stats['conversion_rate'] = group_stats['total_bookings'] / group_stats['total_sessions']

group_stats

,total_bookings,total_sessions,conversion_rate
group,,,
control,12,245,0.048980
treatment,9,256,0.035156


In [ ]:
# test

from scipy import stats

# Split conversion rates by group

control = fake_data[fake_data['group']=='control']['conversion_rate']
treatment = fake_data[fake_data['group']=='treatment']['conversion_rate']

# Run the test
t_stat, p_value = stats.ttest_ind(control, treatment, equal_var=False,alternative="greater")

In [45]:
t_stat,p_value

(-0.39072941161820746, 0.6515750448437726)

In [44]:
fake_data[fake_data['group']=='control']

,user_id,group,bookings,sessions,conversion_rate
0,1,control,0,5,0.000000
1,2,control,1,9,0.111111
3,4,control,0,5,0.000000
5,6,control,0,3,0.000000
9,10,control,0,9,0.000000
13,14,control,0,7,0.000000
14,15,control,1,7,0.142857
15,16,control,0,6,0.000000
18,19,control,0,5,0.000000
19,20,control,0,5,0.000000


In [46]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

# What power did we actually have with our sample?
effect_size = (treatment.mean() - control.mean()) / control.std()

power = analysis.solve_power(
    effect_size=effect_size,
    nobs1=len(control),
    alpha=0.05,
    alternative='larger'
)

print(f"Effect size: {effect_size:.4f}")
print(f"Achieved power: {power:.4f}")

Effect size: 0.0829
Achieved power: 0.1080


In [48]:
required_n = analysis.solve_power(
    effect_size=effect_size,
    power=0.8,
    alpha=0.05,
    alternative='larger'
)

print(f"Required users per group: {required_n:.0f}")

Required users per group: 1799


In [47]:
effect_size

0.08291919570670017